# Monetary Policy Transmission Lags in Canada

**Research Question:** When the Bank of Canada unexpectedly hikes the overnight rate by **+100 bps (+1 percentage point)**, how many quarters does it take for the shock to peak in the **housing market** (fast-moving **CMHC Housing Starts** vs. sticky **NHPI new-home prices**) versus **headline inflation (CPI)** and **real output (GDP)**—and how much macroeconomic volatility is actually driven by monetary policy?

This notebook presents the complete empirical workflow:
1. Historical Canadian macroeconomic & global panel (`1992Q1`–`2026Q2`, 138 quarters)
2. Stationarity diagnostics (ADF & KPSS) across levels and QoQ annualized growth rates
3. **7-Variable Sims (1992) Small-Open-Economy SVARX** with COVID-19 impulse controls (`2020Q2–Q3`) and Pre-COVID (`1992–2019`) benchmark
4. **Dual Housing Channel:** Housing Starts volume (`Q2` trough) vs. NHPI prices (`Q11` trough) vs. Headline CPI (`Q10` trough)
5. **Methodological Triangulation:** SVARX vs. **Jordà (2005) Local Projections** (Newey–West HAC SEs), **US Federal Funds spillover block**, and **alternative Cholesky ordering**
6. Forecast Error Variance Decomposition (FEVD) over 1- to 5-year horizons

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 95
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
PROCESSED = ROOT / "data" / "processed"
RESULTS = PROCESSED / "results"
ROBUST = RESULTS / "robustness"

levels = pd.read_csv(PROCESSED / "quarterly_levels.csv", index_col="quarter")
stationary = pd.read_csv(PROCESSED / "quarterly_stationary.csv", index_col="quarter")
stationarity_df = pd.read_csv(PROCESSED / "stationarity_tests.csv")

print(f"Quarterly panel: {levels.shape[0]} quarters ({levels.index.min()} to {levels.index.max()})")
levels[["real_gdp", "cpi_sa", "housing_starts", "nhpi", "overnight_rate", "usdcad", "bcpi", "us_fedfunds"]].tail()

Quarterly panel: 138 quarters (1992Q1 to 2026Q2)


,real_gdp,cpi_sa,housing_starts,nhpi,overnight_rate,usdcad,bcpi,us_fedfunds
quarter,,,,,,,,
2025Q2,2495975.0,163.500000,281.564000,123.700000,2.750000,1.384067,609.830000,4.330000
2025Q3,2507754.0,164.433333,272.946667,122.966667,2.715909,1.377533,610.390000,4.293333
2025Q4,2501573.0,165.633333,255.484333,122.133333,2.325758,1.394967,605.713333,3.896667
2026Q1,2503604.0,166.666667,241.726667,121.666667,2.250000,1.371533,675.536667,3.640000
2026Q2,2524127.0,168.333333,255.002333,120.800000,2.250000,1.383800,751.973333,3.633333


## 1. Historical Canadian Macro & Housing Overview (1992Q1–2026Q2)

In [2]:
q_dates = pd.PeriodIndex(levels.index, freq="Q").to_timestamp()
s_dates = pd.PeriodIndex(stationary.index, freq="Q").to_timestamp()

fig, axes = plt.subplots(3, 2, figsize=(12.5, 8.2), sharex=True)
axes = axes.flatten()

specs = [
    (q_dates, levels["overnight_rate"], "Bank of Canada Overnight Rate vs. US Fed Funds (%)", "#1e3a8a"),
    (s_dates, stationary["cpi_yoy"], "Headline CPI Inflation (YoY %)", "#2563eb"),
    (q_dates, levels["housing_starts"] / 1000.0, "CMHC Housing Starts (Thousands, SAAR)", "#d97706"),
    (s_dates, stationary["nhpi_yoy"], "New Housing Price Index — NHPI (YoY %)", "#dc2626"),
    (s_dates, stationary["gdp_yoy"], "Real GDP Growth (YoY %)", "#059669"),
    (q_dates, levels["bcpi"], "BoC Commodity Price Index — BCPI (USD)", "#7c3aed"),
]

for i, (dt, ser, title, color) in enumerate(specs):
    ax = axes[i]
    ax.plot(dt, ser, color=color, lw=1.7, label="Canada" if i == 0 else None)
    if i == 0:
        ax.plot(q_dates, levels["us_fedfunds"], color="#64748b", lw=1.2, ls="--", label="US Fed Funds")
    ax.axvspan(pd.Timestamp("2020-01-01"), pd.Timestamp("2021-01-01"), color="gray", alpha=0.16, label="COVID-19" if i == 1 else None)
    if "YoY" in title:
        ax.axhline(0, color="black", lw=0.8, ls="--")
    if "CPI" in title:
        ax.axhline(2.0, color="#dc2626", lw=0.9, ls=":", label="BoC 2% target")
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.grid(alpha=0.25)
    if i in (0, 1):
        ax.legend(fontsize=8)

fig.suptitle("Canadian Inflation-Targeting Macroeconomic Panel (1992Q1–2026Q2)", fontsize=12.5, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 2. Stationarity Diagnostics (ADF & KPSS) & Model Diagnostics

In [3]:
display(stationarity_df[stationarity_df["sample"].str.startswith("Full")][
    ["sample", "variable", "adf_stat", "adf_pvalue", "kpss_stat", "kpss_pvalue", "verdict"]
])

models = ["full_levels_varx", "precovid_levels_varx", "full_stationary_varx", "precovid_stationary_varx"]
diag_rows = []
for m in models:
    meta = json.loads((RESULTS / m / "summary.json").read_text(encoding="utf-8"))
    d = meta["diagnostics"]
    diag_rows.append({
        "model": m,
        "sample": f"{meta['start_quarter']}–{meta['end_quarter']}",
        "T": meta["n_obs"],
        "p": meta["selected_lag"],
        "stable": d["is_stable"],
        "max_eig": d["max_companion_eigenvalue"],
        "whiteness_p": d["whiteness_pvalue"],
        "shock_1sd_bps": round(meta["shock_1sd_pp"] * 100, 1),
    })
pd.DataFrame(diag_rows)

,sample,variable,adf_stat,adf_pvalue,kpss_stat,kpss_pvalue,verdict
0,Full (Levels),log_gdp,-1.6988,0.4318,2.0047,0.0100,Non-stationary
1,Full (Levels),log_cpi,1.1278,0.9954,2.0419,0.0100,Non-stationary
2,Full (Levels),log_nhpi,-0.7389,0.8364,2.0128,0.0100,Non-stationary
3,Full (Levels),log_starts,-2.0791,0.2530,1.2785,0.0100,Non-stationary
4,Full (Levels),overnight_rate,-2.4068,0.1398,1.1108,0.0100,Non-stationary
5,Full (Levels),log_usdcad,-1.7040,0.4291,0.3867,0.0829,"Weak/Borderline (ADF no, KPSS yes)"
6,Full (Levels),log_bcpi,-1.6343,0.4652,1.3247,0.0100,Non-stationary
7,Full (Levels),us_fedfunds,-2.8002,0.0582,0.6556,0.0176,Non-stationary
8,Full (Transformed),gdp_growth,-13.2127,0.0000,0.2361,0.1000,Stationary
9,Full (Transformed),cpi_inflation,-3.5795,0.0062,0.3449,0.1000,Stationary


,model,sample,T,p,stable,max_eig,whiteness_p,shock_1sd_bps
0,full_levels_varx,1992Q1–2026Q2,138,4,True,0.9843,0.0000,38.7
1,precovid_levels_varx,1992Q1–2019Q4,112,4,True,0.9621,0.0016,37.5
2,full_stationary_varx,1992Q2–2026Q2,137,4,True,0.9035,0.0656,41.3
3,precovid_stationary_varx,1992Q2–2019Q4,111,4,True,0.8953,0.0645,40.5


## 3. Dual Housing Channels vs. Headline Inflation & Output (Full vs. Pre-COVID SVARX)

Ordering BoC Commodity Prices (`log_bcpi`) first (Sims, 1992) resolves the short-run CPI price puzzle, while `covid_2020q2` and `covid_2020q3` impulse dummies absorb the pandemic lockdown collapse so the full `1992Q1–2026Q2` sample retains the historic 2022–2024 BoC tightening cycle.

![Dual Housing vs Headline Inflation Summary](../figures/housing_vs_inflation_summary.svg)

In [4]:
def load_bundle(name: str) -> dict[str, pd.DataFrame]:
    base = RESULTS / name
    return {
        "point": pd.read_csv(base / "irf_100bp.csv", index_col="horizon_q"),
        "lo68": pd.read_csv(base / "irf_lower_68.csv", index_col="horizon_q"),
        "hi68": pd.read_csv(base / "irf_upper_68.csv", index_col="horizon_q"),
        "fevd": pd.read_csv(base / "fevd_policy.csv", index_col="horizon_q"),
    }

full_b = load_bundle("full_levels_varx")
pre_b = load_bundle("precovid_levels_varx")
h = full_b["point"].index.values

fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.2))
targets = [
    ("log_starts", "Housing Starts — Volume (% level)"),
    ("log_nhpi", "Housing Prices — NHPI (% level)"),
    ("log_gdp", "Real GDP (% level)"),
    ("log_cpi", "Headline CPI (% level)"),
]

for ax, (col, title) in zip(axes.flatten(), targets):
    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.plot(h, full_b["point"][col], color="#1e3a8a", lw=2.2, label="Full Era SVARX (1992–2026, COVID Ctrl)")
    ax.fill_between(h, full_b["lo68"][col], full_b["hi68"][col], color="#1e3a8a", alpha=0.15)
    ax.plot(h, pre_b["point"][col], color="#dc2626", lw=2.0, ls="--", label="Pre-COVID SVAR (1992–2019)")
    ax.fill_between(h, pre_b["lo68"][col], pre_b["hi68"][col], color="#dc2626", alpha=0.12)
    ax.set_title(title, fontsize=10.5, fontweight="bold")
    ax.set_xlabel("Quarters after +100 bps BoC hike")
    ax.set_xticks(range(0, 21, 4))
    ax.grid(alpha=0.25)
    if col == "log_starts":
        ax.legend(fontsize=8)

axes[0, 0].set_ylabel("% deviation from baseline")
axes[1, 0].set_ylabel("% deviation from baseline")
fig.suptitle("Full-Sample COVID-Controlled SVARX vs. Pre-COVID Benchmark (68% Bootstrap Bands)", fontsize=12, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 4. Methodological Triangulation: Jordà (2005) Local Projections, US Spillovers & Orderings

![Methodological Robustness Triangulation](../figures/robustness_lp_and_ordering.svg)

In [5]:
lp_pt = pd.read_csv(ROBUST / "lp_irf_100bp.csv", index_col="horizon_q")
lp_lo68 = pd.read_csv(ROBUST / "lp_lower_68.csv", index_col="horizon_q")
lp_hi68 = pd.read_csv(ROBUST / "lp_upper_68.csv", index_col="horizon_q")
usff_pt = pd.read_csv(ROBUST / "us_fedfunds_block_irf_100bp.csv", index_col="horizon_q")
fast_pt = pd.read_csv(ROBUST / "alt_fast_housing_irf_100bp.csv", index_col="horizon_q")

fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.2))
for ax, (col, title) in zip(axes.flatten(), targets):
    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.fill_between(h, full_b["lo68"][col], full_b["hi68"][col], color="#2563eb", alpha=0.16, label="SVARX 68% CI")
    ax.plot(h, full_b["point"][col], color="#1e3a8a", lw=2.2, label="7-Var Sims SVARX")
    ax.fill_between(h, lp_lo68[col], lp_hi68[col], color="#dc2626", alpha=0.11, label="LP 68% HAC CI")
    ax.plot(h, lp_pt[col], color="#dc2626", lw=1.9, ls="-.", label="Jordà (2005) LP")
    ax.plot(h, usff_pt[col], color="#7c3aed", lw=1.7, ls="--", label="8-Var (+US Fed Funds)")
    ax.plot(h, fast_pt[col], color="#059669", lw=1.7, ls=":", label="Alt: Fast Housing")
    ax.set_title(title, fontsize=10.5, fontweight="bold")
    ax.set_xlabel("Quarters after +100 bps hike")
    ax.set_xticks(range(0, 21, 4))
    ax.grid(alpha=0.25)
    if col == "log_starts":
        ax.legend(fontsize=7.5, ncol=2)

axes[0, 0].set_ylabel("% deviation from baseline")
axes[1, 0].set_ylabel("% deviation from baseline")
fig.suptitle("Robustness Triangulation: SVARX vs. Local Projections, US Fed Funds & Ordering", fontsize=12, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

pd.read_csv(ROBUST / "robustness_summary.csv")

,method,variable,trough_quarter,trough_response_per_100bp,q4_response,q8_response,q12_response
0,svarx_baseline,log_gdp,10,-0.4956,-0.0576,-0.4466,-0.3922
1,svarx_baseline,log_cpi,10,-0.1759,-0.0079,-0.1507,-0.1625
2,svarx_baseline,log_starts,2,-4.2453,-3.7539,-3.0553,0.3839
3,svarx_baseline,log_nhpi,11,-1.0752,-0.3907,-0.9208,-1.0608
4,jorda_lp,log_gdp,13,-0.8760,-0.2436,-0.4903,-0.4422
5,jorda_lp,log_cpi,7,-0.4944,-0.2884,-0.3603,0.0367
6,jorda_lp,log_starts,12,-5.7270,-4.4363,-3.1707,-5.7270
7,jorda_lp,log_nhpi,20,-1.3307,-0.0513,-0.0901,-0.6830
8,us_fedfunds_block,log_gdp,7,-0.5938,-0.4403,-0.5661,-0.0118
9,us_fedfunds_block,log_cpi,8,-0.0195,0.0598,-0.0195,0.1487


## 5. Forecast Error Variance Decomposition (FEVD) & Peak Summary

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3), sharey=True)
horizons = [4, 8, 12, 20]
x = np.arange(len(horizons))
w = 0.2

for ax, fevd_df, title in [
    (axes[0], full_b["fevd"], "Full Era SVARX (1992–2026, COVID Ctrl)"),
    (axes[1], pre_b["fevd"], "Pre-COVID Benchmark (1992–2019)"),
]:
    ax.bar(x - 1.5 * w, [fevd_df.loc[q, "log_gdp"] for q in horizons], width=w, color="#059669", label="Real GDP")
    ax.bar(x - 0.5 * w, [fevd_df.loc[q, "log_starts"] for q in horizons], width=w, color="#d97706", label="Housing Starts")
    ax.bar(x + 0.5 * w, [fevd_df.loc[q, "log_nhpi"] for q in horizons], width=w, color="#dc2626", label="Housing Prices (NHPI)")
    ax.bar(x + 1.5 * w, [fevd_df.loc[q, "log_cpi"] for q in horizons], width=w, color="#1d4ed8", label="Headline CPI")
    ax.set_title(f"FEVD Share Driven by Policy Shocks\n{title}", fontsize=10, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([f"Q{q}" for q in horizons])
    ax.set_xlabel("Forecast Horizon")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(fontsize=8)

axes[0].set_ylabel("Variance Explained by BoC Rate Shocks (%)")
fig.tight_layout()
plt.show()

peaks = pd.read_csv(RESULTS / "all_models_peak_summary.csv")
peaks[peaks["model"].isin(["full_levels_varx", "precovid_levels_varx"])][
    ["model", "variable", "trough_quarter", "trough_response_per_100bp", "sig_68_negative", "fevd_q4_pct", "fevd_q8_pct", "fevd_q12_pct", "fevd_q20_pct"]
]

,model,variable,trough_quarter,trough_response_per_100bp,sig_68_negative,fevd_q4_pct,fevd_q8_pct,fevd_q12_pct,fevd_q20_pct
0,full_levels_varx,log_bcpi,9,-5.8437,True,0.73,2.25,4.91,5.54
1,full_levels_varx,log_gdp,10,-0.4956,True,0.01,1.16,6.20,5.50
2,full_levels_varx,log_cpi,10,-0.1759,False,0.04,0.22,0.85,0.96
3,full_levels_varx,log_starts,2,-4.2453,True,5.64,7.91,8.54,7.90
4,full_levels_varx,log_nhpi,11,-1.0752,True,0.34,1.64,3.22,4.49
5,full_levels_varx,log_usdcad,1,0.6009,False,0.87,5.08,8.36,8.51
6,precovid_levels_varx,log_bcpi,8,-6.4108,True,0.40,2.46,6.34,7.54
7,precovid_levels_varx,log_gdp,10,-0.4931,True,3.46,4.74,9.66,9.76
8,precovid_levels_varx,log_cpi,9,-0.1528,True,0.44,1.14,2.72,4.69
9,precovid_levels_varx,log_starts,2,-2.1569,True,1.50,1.04,1.48,1.47
